In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('AnimeRecommender') \
    .config('spark.ui.port', '4040') \
    .getOrCreate()

print(f'Spark version: {spark.version}')
print('Spark UI: http://localhost:4040')

Spark version: 3.5.0
Spark UI: http://localhost:4040


In [2]:
BASE = '/home/jovyan/work/data/preprocessed'

reviews_data = spark.read.csv(f'{BASE}/reviews.csv', header=True, inferSchema=True)
anime_data   = spark.read.csv(f'{BASE}/animes.csv',    header=True, inferSchema=True)

print('reviews schema:')
reviews_data.printSchema()
print('animes schema:')
anime_data.printSchema()

reviews schema:
root
 |-- _c0: integer (nullable = true)
 |-- uid: integer (nullable = true)
 |-- profile: string (nullable = true)
 |-- anime_id: integer (nullable = true)
 |-- score: integer (nullable = true)

animes schema:
root
 |-- _c0: integer (nullable = true)
 |-- anime_id: integer (nullable = true)
 |-- genre: string (nullable = true)



In [3]:
from pyspark.ml.feature import StringIndexer
review_indexed = StringIndexer(inputCol="profile", outputCol="user_id")
reviews_data_indexed = review_indexed.fit(reviews_data).transform(reviews_data)

final_data = reviews_data_indexed.select(
    col('user_id').cast('integer'),
    col('anime_id').cast('integer'),
    col('score').cast('float')
).filter(col('score') > 0).filter( col('score') <= 10)

print(f'Total rated reviews: {final_data.count()}')
final_data.show(10)

Total rated reviews: 130,517
+-------+--------+-----+
|user_id|anime_id|score|
+-------+--------+-----+
|     32|   34096|  8.0|
|   1104|   34599| 10.0|
|   1825|   28891|  7.0|
|   3796|    2904|  9.0|
|   9589|    4181| 10.0|
|   9872|    2904| 10.0|
|    554|   16664|  6.0|
+-------+--------+-----+
only showing top 7 rows



In [4]:
anime_data.select('anime_id', 'genre').show(5, truncate=False)

+--------+-------------------------------------------------------------------------------------+
|anime_id|genre                                                                                |
+--------+-------------------------------------------------------------------------------------+
|28891   |['Comedy', 'Sports', 'Drama', 'School', 'Shounen']                                   |
|23273   |['Drama', 'Music', 'Romance', 'School', 'Shounen']                                   |
|34599   |['Sci-Fi', 'Adventure', 'Mystery', 'Drama', 'Fantasy']                               |
|5114    |['Action', 'Military', 'Adventure', 'Comedy', 'Drama', 'Magic', 'Fantasy', 'Shounen']|
|31758   |['Action', 'Mystery', 'Supernatural', 'Vampire']                                     |
+--------+-------------------------------------------------------------------------------------+
only showing top 5 rows



In [12]:
from pyspark.ml.feature import CountVectorizer, IDF, Tokenizer
from pyspark.sql.functions import split, lower, trim, col, regexp_replace

# Clean and split the genre string into an array of tokens
anime_clean = anime_data \
    .withColumnRenamed('anime_id', 'anime_id') \
    .withColumn('genre_stripped',
        regexp_replace(col('genre'), r"[\[\]']", '')  # remove [ ] and '
    ) \
    .withColumn('genres_array',
        split(lower(trim(col('genre_stripped'))), ',\\s*')
    ) \
    .drop('genre_stripped') \
    .dropna(subset=['genre'])

anime_clean.select('anime_id', 'genre', 'genres_array').show(5, truncate=False)

+--------+-------------------------------------------------------------------------------------+---------------------------------------------------------------------+
|anime_id|genre                                                                                |genres_array                                                         |
+--------+-------------------------------------------------------------------------------------+---------------------------------------------------------------------+
|28891   |['Comedy', 'Sports', 'Drama', 'School', 'Shounen']                                   |[comedy, sports, drama, school, shounen]                             |
|23273   |['Drama', 'Music', 'Romance', 'School', 'Shounen']                                   |[drama, music, romance, school, shounen]                             |
|34599   |['Sci-Fi', 'Adventure', 'Mystery', 'Drama', 'Fantasy']                               |[sci-fi, adventure, mystery, drama, fantasy]                         

In [28]:
from pyspark.ml.feature import CountVectorizer, IDF
from pyspark.ml import Pipeline

cv  = CountVectorizer(inputCol='genres_array', outputCol='tf')
idf = IDF(inputCol='tf', outputCol='tfidf')

pipeline = Pipeline(stages=[cv, idf])
anime_vectorized = pipeline.fit(anime_clean).transform(anime_clean)

anime_vectorized.select('anime_id', 'genres_array','tfidf').show(5, truncate=False)

+--------+---------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|anime_id|genres_array                                                         |tfidf                                                                                                                                                                           |
+--------+---------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|28891   |[comedy, sports, drama, school, shounen]                             |(44,[0,6,7,11,19],[1.0559987199793983,1.8867120635933943,2.1358203951885684,2.3368971113633537,3.171722555549222])                                

In [33]:
from pyspark.ml.feature import CountVectorizer
from pyspark.sql.functions import least, lit

cv = CountVectorizer(inputCol='genres_array', outputCol='tfidf', binary=True)
anime_vectorized = cv.fit(anime_clean).transform(anime_clean)
anime_vectorized.select('anime_id', 'genres_array','tfidf').show(5, truncate=False)

+--------+---------------------------------------------------------------------+----------------------------------------------------------+
|anime_id|genres_array                                                         |tfidf                                                     |
+--------+---------------------------------------------------------------------+----------------------------------------------------------+
|28891   |[comedy, sports, drama, school, shounen]                             |(44,[0,6,7,11,19],[1.0,1.0,1.0,1.0,1.0])                  |
|23273   |[drama, music, romance, school, shounen]                             |(44,[6,7,8,9,11],[1.0,1.0,1.0,1.0,1.0])                   |
|34599   |[sci-fi, adventure, mystery, drama, fantasy]                         |(44,[2,3,4,6,20],[1.0,1.0,1.0,1.0,1.0])                   |
|5114    |[action, military, adventure, comedy, drama, magic, fantasy, shounen]|(44,[0,1,2,3,6,7,16,24],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])|
|31758   |[action, m

In [34]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Pull vectors to driver — fine for a few thousand anime
anime_pd = anime_vectorized.select('anime_id', 'tfidf').toPandas()
anime_pd['vec'] = anime_pd['tfidf'].apply(lambda v: v.toArray())

# Build matrix and compute all pairwise similarities at once
matrix = np.vstack(anime_pd['vec'].values)
sim_matrix = cosine_similarity(matrix)  # shape: (n_anime, n_anime)

# Map index → anime_id
idx_to_uid = anime_pd['anime_id'].to_dict()
uid_to_idx = {v: k for k, v in idx_to_uid.items()}

In [36]:
def content_recommend(anime_id, n=10):
    if anime_id not in uid_to_idx:
        print(f'anime_id {anime_id} not found')
        return

    idx = uid_to_idx[anime_id]
    scores = sim_matrix[idx]

    # Get top-n most similar (excluding itself)
    top_indices = np.argsort(scores)[::-1][1:n+1]
    
    results = pd.DataFrame({
        'anime_id':  [idx_to_uid[i] for i in top_indices],
        'similarity': [scores[i]     for i in top_indices]
    })
    
    # Join genre info back
    genre_map = anime_pd.set_index('anime_id')['tfidf']  # reuse anime_pd
    genre_lookup = anime_clean.select('anime_id', 'genre').toPandas() \
                              .set_index('anime_id')
    results['genre'] = results['anime_id'].map(genre_lookup['genre'])
    
    return results

# Test with first anime in dataset
sample_uid = anime_pd['anime_id'].iloc[0]
print(f'Recommendations similar to anime {sample_uid}:')
content_recommend(sample_uid)

Recommendations similar to anime 28891:


,anime_id,similarity,genre
0,29755,1.0,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun..."
1,35110,1.0,"['Comedy', 'Drama', 'School', 'Shounen', 'Spor..."
2,40262,1.0,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun..."
3,35111,1.0,"['Comedy', 'Drama', 'School', 'Shounen', 'Spor..."
4,37403,1.0,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun..."
5,38883,1.0,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun..."
6,40776,1.0,"['Comedy', 'Drama', 'School', 'Shounen', 'Spor..."
7,170,1.0,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun..."
8,20583,1.0,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun..."
9,9077,1.0,"['Sports', 'Comedy', 'School', 'Drama', 'Shoun..."


In [31]:
from pyspark.ml.recommendation import ALSModel
model = ALSModel.load('/home/jovyan/work/models/user-item')

In [32]:
def hybrid_recommend_v2(user_id, n=10, als_weight=0.5, content_weight=0.5):
    # --- ALS user-item score ---
    target_user = spark.createDataFrame([(user_id,)], ['user_id'])
    als_recs = model.recommendForUserSubset(target_user, 50) \
        .select('user_id', explode('recommendations').alias('rec')) \
        .select('user_id', col('rec.anime_id'), col('rec.rating').alias('als_score')) \
        .toPandas()

    if als_recs.empty:
        print('No ALS recommendations for this user')
        return

    # --- Content similarity to user's rated anime ---
    user_history = final_data.filter(col('user_id') == user_id) \
        .select('anime_id').toPandas()['anime_id'].tolist()

    def avg_content_sim(candidate_uid):
        if candidate_uid not in uid_to_idx:
            return 0.0
        c_idx = uid_to_idx[candidate_uid]
        sims = [
            sim_matrix[uid_to_idx[h]][c_idx]
            for h in user_history if h in uid_to_idx
        ]
        return float(np.mean(sims)) if sims else 0.0

    als_recs['content_score'] = als_recs['anime_id'].map(avg_content_sim)

    # --- Normalize + blend ---
    for col_name in ['als_score', 'content_score']:
        mn, mx = als_recs[col_name].min(), als_recs[col_name].max()
        als_recs[f'{col_name}_norm'] = (als_recs[col_name] - mn) / (mx - mn + 1e-9)

    als_recs['hybrid_score'] = (
        als_weight    * als_recs['als_score_norm'] +
        content_weight * als_recs['content_score_norm']
    )

    result = als_recs.sort_values('hybrid_score', ascending=False).head(n)

    # Join genre labels
    genre_lookup = anime_clean.select('anime_id', 'genre').toPandas() \
                              .set_index('anime_id')
    result['genre'] = result['anime_id'].map(genre_lookup['genre'])

    return result[['anime_id', 'genre', 'als_score', 'content_score', 'hybrid_score']]

# Test
target_uid = final_data.select('user_id').first()[0]
hybrid_recommend_v2(target_uid)

,anime_id,genre,als_score,content_score,hybrid_score
1,36885,"['Comedy', 'Ecchi', 'Harem', 'Romance', 'School']",14.139758,0.194189,0.620262
42,38753,"['Comedy', 'Drama', 'Romance', 'School', 'Shou...",10.546517,0.284827,0.511710
0,36122,['Hentai'],17.101545,0.000000,0.500000
3,37403,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun...",13.240625,0.161966,0.496720
4,31165,['Drama'],13.180426,0.149472,0.470303
23,33161,"['Comedy', 'Romance', 'School']",11.063675,0.237619,0.467362
21,3467,"['Comedy', 'Romance']",11.103932,0.225940,0.449858
22,35988,"['Adventure', 'Comedy', 'Romance', 'Fantasy']",11.073065,0.220166,0.437423
36,2523,"['Adventure', 'Drama', 'Fantasy', 'Martial Art...",10.737107,0.220753,0.413429
6,34928,"['Sci-Fi', 'Comedy']",12.750710,0.125603,0.396392
